# Notebook 2: Computing Convex Hulls - Part 1

**Atlas source reading copy · local-server**

[Course guide](index.ipynb) · [Pinned original](https://github.com/PHI-Case-Studies/1854-Cholera-Outbreak-London-Advanced-2/blob/d1b56c135b0fab2f470b837a51e6777dfc20b286/NB02-Computing-Convex-Hulls.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Use the source local/server environment for native GIS, shell, installed project or legacy dependencies; Lab provides a reading/editing copy. Follow notebook order to regenerate omitted outputs; optional unlicensed media are external continuations.

**Provenance:** `NB02-Computing-Convex-Hulls.ipynb` at `d1b56c135b0fab2f470b837a51e6777dfc20b286`; original SHA-256 `cc011a921ceddd3ee625c196a31ad09d6e22714322fcdb9ba27e3ac667f62d85`. Source-specific license/notices retained under notices; data and media have separate terms.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


In [ ]:
%run resources/library.py

# Notebook 2: Computing Convex Hulls - Part 1

## What is a Convex Hull?

To be able to create Voronoi tesselations, we must be able to create convex hulls.

From this [Medium Blog article](https://medium.com/@pascal.sommer.ch/a-gentle-introduction-to-the-convex-hull-problem-62dfcabee90c) a convex hull is 
> "a set of points is defined as the smallest convex polygon, that encloses all of the points in the set. Convex means that the polygon has no corner that is bent inwards."

### Methods to compute convex hulls (visual explanation)

Here is a YouTube below that explains three methods that can be used to compute convex hulls:
1. Jarvis March
2. Graham Scan
3. Chan's Algorithm

In [ ]:
show_youtube(url='https://www.youtube.com/embed/ZnTiWcIznEQ', width=900, height=500)

### Using the `scipy.spatial` library

The Quickhull Algorithm is another method to compute convex hulls. We can add that as a fourth method:

4. Quickhull Algorithm

We will use the `scipy.spatial` library to plot a convex hull from random points.

In [ ]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np

In [ ]:
random_2D_array = np.random.random_sample((50,2))
hull = ConvexHull(random_2D_array)
sample_hull_plot = convex_hull_plot_2d(hull)

You can re-run the code cell again to generate more random convex hull plots. 

You can also review the Scipy Spatial documentation under **References** for more details about the `scipy.spatial` library.

## Load Cholera data

In [ ]:
import pandas as pd

pd.__version__

In [ ]:
pumps_df = pd.read_csv('data/pumps.csv')

pumps_df

In [ ]:
deaths_df = pd.read_csv('data/cholera_deaths.csv')

deaths_df

## Prepare data for computing convex hull

In [ ]:
points_pumps = np.array([[lat,lon] for lat,lon in pumps_df[['LAT','LON']].values])

points_pumps

In [ ]:
points_deaths = np.array([[lat,lon] for lat,lon in deaths_df[['LAT','LON']].values])

points_deaths

### Combine all points

In [ ]:
points = np.concatenate((points_pumps, points_deaths), axis=0)

points[3,0], points[3,1]

## Compute Convex Hull

In [ ]:
from scipy.spatial import ConvexHull, Voronoi

In [ ]:
hull = ConvexHull(points)

hull

In [ ]:
hull.vertices

In [ ]:
hull.simplices

In [ ]:
for simplex in hull.simplices:
    print("simplex, indices of points:", simplex)
    print ("points, 0 and 1:", points[simplex,0], points[simplex,1])

In [ ]:
x=[]
y=[]
for vertex in hull.vertices:
    y.append(points[vertex,0])
    x.append(points[vertex,1])
    
x,y

In [ ]:
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
plt.plot(points[:,1], points[:,0], 'o')
for simplex in hull.simplices:
    plt.plot(points[simplex,1], points[simplex,0], 'k-')
    
plt.show()

### Solve for vertices (points) of convex hull

In [ ]:
from shapely.geometry import Point, MultiPoint, Polygon

In [ ]:
points[vertex,0]

In [ ]:
vertex_points = MultiPoint([(points[vertex,1], points[vertex,0]) \
                            for vertex in hull.vertices])

vertex_points

### Convert vertices to a Shapely Polygon and apply `.buffer()` function

In [ ]:
convexhull_poly = Polygon([[p.x,p.y] for p in vertex_points.geoms]).buffer(.002)

convexhull_poly

Recall the buffer function we used in isochrone mapping.

### Convert convex hull polygon to Geopandas geodataframe

In [ ]:
import geopandas as gpd

gpd.__version__

In [ ]:
crs = {'init': 'epsg:4326'}
convexhull_gdf = gpd.GeoDataFrame(index=[0], crs=crs, geometry=[convexhull_poly])

convexhull_gdf

Recall the CRS function we used for isochrone mapping. 

In [ ]:
convexhull_gdf.plot(alpha=0.2, edgecolor='black',linewidth=2)

### Save the convex hull polygon

In [ ]:
convexhull_gdf.to_file('outputs/convexhull.shp')

convexhull_gdf

### Alternative method to display convex hull

In [ ]:
x,y = convexhull_poly.exterior.xy
plt.plot(x,y, color='gray')

## Bring all the plots together

### Plot pumps

In [ ]:
plt.plot(points_pumps[:,1], points_pumps[:,0], 'o', color='red', )

### Plot death locations

In [ ]:
plt.plot(points_deaths[:,1], points_deaths[:,0], 'o')

### Combined plots: Minimum Viable Product

In [ ]:
my_dpi=96

fig, ax = plt.subplots(figsize=((432*1.8)/my_dpi, (288*1.8)/my_dpi), dpi=my_dpi)

# plot death locations, color this grey
ax.plot(points_deaths[:,1], points_deaths[:,0], 'o', color='grey')

# plot pump locations, color this red
ax.plot(points_pumps[:,1], points_pumps[:,0], 'o', color='red', )

# plot convex hull polygon
x,y = convexhull_poly.exterior.xy
ax.plot(x,y, color='gray')

# set title and axis labels
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Pumps and Deaths with Alpha Shape, Soho District')

With this Python approach, we know we can use the convex hull to constrain potential infinite vertices when computing Voronoi polygons.

## Alternative Method for Computing Convex Hull - `alphashape`

See Notebook 3 for an alternative method to compute convex hull using the `alphashape` package.

## References

1. Convex Hull Reference: https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.ConvexHull.html
2. Scipy Spatial Tutorial (includes convex hull): https://docs.scipy.org/doc/scipy/reference/tutorial/spatial.html
3. https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.convex_hull_plot_2d.html#scipy.spatial.convex_hull_plot_2d
4. Numpy concatenate arrays: http://lagrange.univ-lyon1.fr/docs/numpy/1.11.0/reference/generated/numpy.concatenate.html